# Mini Project 01 — Notebook 02: Phát hiện đối tượng với E-ELAN (Task 3)

**Nhóm:** Nguyễn Đức Quang — 23001918 · Hà Minh Quang — 23001916

**Kiến trúc:** `EELANBackbone` (nạp từ `backbone.pt` của Notebook 01, đóng băng) → **Neck = SimpleFPN** → **Head = SSDHead** (anchor-based, one-stage) — tất cả module lấy từ Lab 05.

**Quy trình:**
1. Xây dữ liệu nhỏ 3 lớp **cat / dog / panda**, chuẩn hóa về **1 định dạng YOLO** thống nhất.
2. Ghép Backbone + Neck + Head, sinh anchor đa tỉ lệ (P3/P4/P5).
3. Huấn luyện bằng **Multibox loss** (SmoothL1 + CrossEntropy + hard negative mining).
4. Lưu `weights/best_det.pt`.
5. Inference (decode + NMS) → vẽ bbox, xuất CSV (bbox, class, confidence).
6. Đánh giá **mAP@0.5**, precision/recall.


## 0. Môi trường & đường dẫn

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/MiniProject01'
    IN_COLAB = True
except Exception:
    ROOT = r'D:/CV nâng cao/Project/Mini Project 01'
    IN_COLAB = False

DATA_DET = os.path.join(ROOT, 'data', 'detection')
WEIGHTS  = os.path.join(ROOT, 'weights')
RESULTS  = os.path.join(ROOT, 'results')
for d in [DATA_DET, WEIGHTS, RESULTS,
          os.path.join(DATA_DET,'images','train'), os.path.join(DATA_DET,'images','val'),
          os.path.join(DATA_DET,'labels','train'), os.path.join(DATA_DET,'labels','val')]:
    os.makedirs(d, exist_ok=True)
print('IN_COLAB=', IN_COLAB, '| DATA_DET=', DATA_DET)

In [ ]:
# !pip -q install torchmetrics  # (không bắt buộc; mAP tự cài đặt bên dưới)
import math, time, glob, csv, random, xml.etree.ElementTree as ET
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print('Device:', device)

## 1. CONFIG

In [ ]:
CLASSES = ['cat', 'dog', 'panda']          # id 0,1,2  (lớp 0 của head = background)
NUM_CLASSES = len(CLASSES)                  # số lớp đối tượng (không tính background)
NAME2ID = {c: i for i, c in enumerate(CLASSES)}

CFG = {
    'img_size': 320,                        # ảnh vuông 320x320 -> feat 40/20/10
    'strides': [8, 16, 32],                 # P3, P4, P5
    'anchor_bases': [32, 80, 160],          # kích thước anchor cơ sở (px) cho P3/P4/P5
    'anchor_scales': [1.0, 1.4],            # 2 scale
    'anchor_ratios': [1.0, 2.0, 0.5],       # 3 tỉ lệ  -> 6 anchor/cell
    'variances': (0.1, 0.2),

    'pos_iou': 0.5, 'neg_iou': 0.4,
    'neg_pos_ratio': 3,

    'batch_size': 16, 'epochs': 30, 'lr': 2e-3, 'weight_decay': 5e-4,
    'num_workers': 2,

    'freeze_backbone': True,                # đóng băng backbone đã train ở NB01 (theo gợi ý đề)
    'backbone_ckpt': os.path.join(WEIGHTS, 'backbone.pt'),
    'best_ckpt':     os.path.join(WEIGHTS, 'best_det.pt'),

    'conf_th': 0.3, 'nms_iou': 0.45,        # ngưỡng inference
    'val_ratio': 0.15,
}
NUM_ANCHORS = len(CFG['anchor_scales']) * len(CFG['anchor_ratios'])   # 6
print('num_anchors/cell =', NUM_ANCHORS)

## 2. Xây dựng & chuẩn hóa dữ liệu
Chuẩn đầu ra thống nhất — **định dạng YOLO**:
```
data/detection/images/{train,val}/*.jpg
data/detection/labels/{train,val}/*.txt   # mỗi dòng: class_id cx cy w h  (chuẩn hóa 0..1)
data/detection/classes.txt
```
- **cat, dog:** tự động tải từ **Pascal VOC 2007** (torchvision), lọc ảnh chứa cat/dog.
- **panda:** tải từ **Roboflow** (YOLO export) rồi nạp bằng `ingest_yolo_folder` (cần API key / tải tay).

In [ ]:
# Tiện ích ghi 1 mẫu ra định dạng unified YOLO
def write_sample(split, img_pil, boxes_xyxy, labels, stem):
    """boxes_xyxy: list [x1,y1,x2,y2] theo pixel; labels: list class_id."""
    W, H = img_pil.size
    img_dir = os.path.join(DATA_DET, 'images', split)
    lab_dir = os.path.join(DATA_DET, 'labels', split)
    img_path = os.path.join(img_dir, stem + '.jpg')
    img_pil.convert('RGB').save(img_path, quality=92)
    lines = []
    for (x1,y1,x2,y2), c in zip(boxes_xyxy, labels):
        cx = ((x1+x2)/2)/W; cy = ((y1+y2)/2)/H
        bw = (x2-x1)/W;     bh = (y2-y1)/H
        cx=min(max(cx,0),1); cy=min(max(cy,0),1); bw=min(max(bw,0),1); bh=min(max(bh,0),1)
        if bw<=0 or bh<=0: continue
        lines.append(f'{c} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    with open(os.path.join(lab_dir, stem + '.txt'), 'w') as f:
        f.write('\n'.join(lines))

with open(os.path.join(DATA_DET, 'classes.txt'), 'w') as f:
    f.write('\n'.join(CLASSES))
print('classes.txt:', CLASSES)

In [ ]:
# ---- 2a. Pascal VOC 2007 -> cat/dog ----
BUILD_VOC = True   # đặt False nếu đã build xong (tránh làm lại)

if BUILD_VOC:
    VOC_ROOT = os.path.join(DATA_DET, '_voc_cache')
    os.makedirs(VOC_ROOT, exist_ok=True)
    voc = datasets.VOCDetection(VOC_ROOT, year='2007', image_set='trainval', download=True)
    want = {'cat', 'dog'}
    n_written = 0
    for k in range(len(voc)):
        img, target = voc[k]
        objs = target['annotation']['object']
        if isinstance(objs, dict): objs = [objs]
        boxes, labels = [], []
        for o in objs:
            name = o['name']
            if name in want:
                b = o['bndbox']
                x1,y1,x2,y2 = float(b['xmin']),float(b['ymin']),float(b['xmax']),float(b['ymax'])
                boxes.append([x1,y1,x2,y2]); labels.append(NAME2ID[name])
        if boxes:
            split = 'val' if (k % 100) < int(CFG['val_ratio']*100) else 'train'
            write_sample(split, img, boxes, labels, f'voc_{k:05d}')
            n_written += 1
    print(f'VOC cat/dog: đã ghi {n_written} ảnh.')
else:
    print('Bỏ qua build VOC.')

In [ ]:
# ---- 2b. Panda (Roboflow YOLO export) ----
# Cách lấy: roboflow.com -> tìm dataset "panda" có bbox -> Download -> định dạng YOLOv8 ->
# giải nén vào 1 thư mục, trỏ PANDA_SRC tới đó (có images/ và labels/).
#
# Hoặc dùng pip (điền API key):
# !pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# ds = rf.workspace("ws").project("panda-xxxx").version(1).download("yolov8",
#        location=os.path.join(DATA_DET,'_panda_raw'))
# PANDA_SRC = ds.location

def ingest_yolo_folder(src_root, out_class_id=2, val_ratio=0.15, prefix='panda'):
    """Nạp 1 thư mục YOLO (mọi class trong đó -> gán thành out_class_id vì dataset panda-only)."""
    img_files = []
    for ext in ('*.jpg','*.jpeg','*.png','*.JPG','*.PNG'):
        img_files += glob.glob(os.path.join(src_root, '**', ext), recursive=True)
    img_files = sorted(img_files)
    n = 0
    for i, imgf in enumerate(img_files):
        labf = imgf.replace('/images/', '/labels/')
        labf = os.path.splitext(labf)[0] + '.txt'
        if not os.path.exists(labf):
            # thử cùng thư mục
            alt = os.path.splitext(imgf)[0] + '.txt'
            labf = alt if os.path.exists(alt) else None
        if labf is None: continue
        try:
            im = Image.open(imgf).convert('RGB')
        except Exception:
            continue
        W, Hh = im.size
        boxes, labels = [], []
        with open(labf) as f:
            for line in f:
                p = line.split()
                if len(p) < 5: continue
                _, cx, cy, bw, bh = p[:5]
                cx,cy,bw,bh = map(float,(cx,cy,bw,bh))
                x1=(cx-bw/2)*W; y1=(cy-bh/2)*Hh; x2=(cx+bw/2)*W; y2=(cy+bh/2)*Hh
                boxes.append([x1,y1,x2,y2]); labels.append(out_class_id)
        if boxes:
            split = 'val' if (i % 100) < int(val_ratio*100) else 'train'
            write_sample(split, im, boxes, labels, f'{prefix}_{i:05d}')
            n += 1
    print(f'Panda: đã ghi {n} ảnh từ {src_root}')
    return n

# >>> CHỈNH đường dẫn tới thư mục panda đã tải rồi bỏ comment dòng dưới:
# ingest_yolo_folder(os.path.join(DATA_DET, '_panda_raw'), out_class_id=NAME2ID['panda'])
print('Nhớ nạp panda trước khi train (xem hướng dẫn trong cell).')

In [ ]:
# ---- 2c. Thống kê & trực quan vài mẫu ----
def count_split(split):
    labs = glob.glob(os.path.join(DATA_DET,'labels',split,'*.txt'))
    per = defaultdict(int); nimg = 0
    for l in labs:
        lines = [x for x in open(l).read().splitlines() if x.strip()]
        if lines: nimg += 1
        for x in lines: per[int(x.split()[0])] += 1
    return nimg, per

for sp in ['train','val']:
    nimg, per = count_split(sp)
    print(f'[{sp}] images={nimg} | boxes/class=' + ', '.join(f'{CLASSES[k]}:{per[k]}' for k in range(NUM_CLASSES)))

## 3. Dataset & DataLoader (resize 320×320, scale bbox)

In [ ]:
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

class YoloDetDataset(Dataset):
    def __init__(self, split, img_size=320, train=True):
        self.img_files = sorted(sum([glob.glob(os.path.join(DATA_DET,'images',split,e))
                                     for e in ('*.jpg','*.png','*.jpeg')], []))
        self.split = split; self.img_size = img_size; self.train = train
    def __len__(self): return len(self.img_files)
    def __getitem__(self, i):
        imgf = self.img_files[i]
        im = Image.open(imgf).convert('RGB')
        W, H = im.size
        labf = imgf.replace(os.sep+'images'+os.sep, os.sep+'labels'+os.sep)
        labf = os.path.splitext(labf)[0] + '.txt'
        boxes, labels = [], []
        if os.path.exists(labf):
            for line in open(labf):
                p = line.split()
                if len(p) < 5: continue
                c, cx, cy, bw, bh = int(p[0]), *map(float, p[1:5])
                x1=(cx-bw/2)*W; y1=(cy-bh/2)*H; x2=(cx+bw/2)*W; y2=(cy+bh/2)*H
                boxes.append([x1,y1,x2,y2]); labels.append(c)
        S = self.img_size
        # augment: random horizontal flip
        do_flip = self.train and random.random() < 0.5
        im = im.resize((S, S), Image.BILINEAR)
        sx, sy = S / W, S / H
        bx = []
        for (x1,y1,x2,y2) in boxes:
            nx1,ny1,nx2,ny2 = x1*sx, y1*sy, x2*sx, y2*sy
            if do_flip: nx1, nx2 = S-nx2, S-nx1
            bx.append([nx1,ny1,nx2,ny2])
        if do_flip: im = im.transpose(Image.FLIP_LEFT_RIGHT)
        t = transforms.functional.to_tensor(im)
        t = (t - MEAN) / STD
        boxes_t = torch.tensor(bx, dtype=torch.float32) if bx else torch.zeros((0,4))
        labels_t = torch.tensor(labels, dtype=torch.long) if labels else torch.zeros((0,), dtype=torch.long)
        return t, boxes_t, labels_t, imgf

def collate(batch):
    imgs = torch.stack([b[0] for b in batch])
    boxes = [b[1] for b in batch]; labels = [b[2] for b in batch]; paths=[b[3] for b in batch]
    return imgs, boxes, labels, paths

train_ds = YoloDetDataset('train', CFG['img_size'], train=True)
val_ds   = YoloDetDataset('val',   CFG['img_size'], train=False)
print('train:', len(train_ds), '| val:', len(val_ds))
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], collate_fn=collate)

## 4. Backbone E-ELAN (định nghĩa lại giống NB01 để nạp weights)

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, c_in, c_out, k=3, s=1, p=None, g=1, act=True):
        super().__init__()
        if p is None: p=(k-1)//2
        self.conv=nn.Conv2d(c_in,c_out,k,s,p,groups=g,bias=False)
        self.bn=nn.BatchNorm2d(c_out)
        self.act=nn.SiLU(inplace=True) if act else nn.Identity()
    def forward(self,x): return self.act(self.bn(self.conv(x)))

def channel_shuffle(x, groups):
    if groups<=1: return x
    b,c,h,w=x.shape; x=x.view(b,groups,c//groups,h,w)
    return x.transpose(1,2).contiguous().view(b,c,h,w)

class EELANLite(nn.Module):
    def __init__(self, C_in, C_out=None, m=2.0, g=2, branch_depths=None, use_skip=True):
        super().__init__()
        self.C_in=C_in; self.C_out=C_in if C_out is None else C_out; self.g=g
        self.use_skip=use_skip and (self.C_out==self.C_in)
        C_exp=int(round(m*C_in)); C_exp=max(4*g,(C_exp//(4*g))*(4*g)); self.C_exp=C_exp
        self.expand=ConvBNAct(C_in,C_exp,1,1,0,g=g)
        self.num_branches=4; self.split_ch=C_exp//4
        if branch_depths is None: branch_depths=[0,1,2,3]
        self.branch_depths=branch_depths[:4]
        br=[]
        for d in self.branch_depths:
            layers=[ConvBNAct(self.split_ch,self.split_ch,3,1,1,g=g) for _ in range(d)]
            br.append(nn.Sequential(*layers) if layers else nn.Identity())
        self.branches=nn.ModuleList(br)
        self.fuse=ConvBNAct(C_exp,self.C_out,1,1,0,g=1)
    def forward(self,x):
        idt=x; y=self.expand(x); chunks=torch.chunk(y,4,dim=1)
        outs=[channel_shuffle(b(c),self.g) for b,c in zip(self.branches,chunks)]
        y=self.fuse(torch.cat(outs,1))
        if self.use_skip and y.shape==idt.shape: y=y+idt
        return y

class EELANBackbone(nn.Module):
    def __init__(self, C_in=3, C_stem=32, widths=(64,128,192,256), m=2.0, g=2):
        super().__init__()
        w1,w2,w3,w4=widths
        self.stem=nn.Sequential(ConvBNAct(C_in,C_stem,3,2), ConvBNAct(C_stem,C_stem,3,1))
        self.stage1=nn.Sequential(ConvBNAct(C_stem,w1,3,2), EELANLite(w1,w1,m,g))
        self.stage2=nn.Sequential(ConvBNAct(w1,w2,3,2), EELANLite(w2,w2,m,g))
        self.stage3=nn.Sequential(ConvBNAct(w2,w3,3,2), EELANLite(w3,w3,m,g))
        self.stage4=nn.Sequential(ConvBNAct(w3,w4,3,2), EELANLite(w4,w4,m,g))
        self.out_channels={'C3':w2,'C4':w3,'C5':w4}
    def forward(self,x):
        x=self.stem(x); x=self.stage1(x)
        c3=self.stage2(x); c4=self.stage3(c3); c5=self.stage4(c4)
        return c3,c4,c5

## 5. Neck (SimpleFPN) + Head (SSDHead) — từ Lab 05, ghép thành Detector

In [ ]:
class SimpleFPN(nn.Module):
    def __init__(self, c3, c4, c5, out_channels=256):
        super().__init__()
        self.lateral3=nn.Conv2d(c3,out_channels,1)
        self.lateral4=nn.Conv2d(c4,out_channels,1)
        self.lateral5=nn.Conv2d(c5,out_channels,1)
        self.smooth3=nn.Conv2d(out_channels,out_channels,3,padding=1)
        self.smooth4=nn.Conv2d(out_channels,out_channels,3,padding=1)
    def forward(self, C3, C4, C5):
        P5=self.lateral5(C5)
        P4=self.lateral4(C4)+F.interpolate(P5,size=C4.shape[-2:],mode='nearest')
        P3=self.lateral3(C3)+F.interpolate(P4,size=C3.shape[-2:],mode='nearest')
        return self.smooth3(P3), self.smooth4(P4), P5

class SSDHead(nn.Module):
    """Head dùng chung cho các mức, trả về cls[B,A,K] và reg[B,A,4] đã nối các mức."""
    def __init__(self, in_channels, num_anchors, num_classes_bg):
        super().__init__()
        self.K=num_classes_bg; self.na=num_anchors
        self.cls=nn.Conv2d(in_channels, num_anchors*num_classes_bg, 3, padding=1)
        self.reg=nn.Conv2d(in_channels, num_anchors*4, 3, padding=1)
    def forward(self, feats):
        cls_all, reg_all = [], []
        for f in feats:
            B=f.shape[0]
            c=self.cls(f).permute(0,2,3,1).contiguous().view(B,-1,self.K)
            r=self.reg(f).permute(0,2,3,1).contiguous().view(B,-1,4)
            cls_all.append(c); reg_all.append(r)
        return torch.cat(cls_all,1), torch.cat(reg_all,1)

class EELANDetector(nn.Module):
    def __init__(self, backbone, num_anchors, num_classes, fpn_out=256):
        super().__init__()
        self.backbone=backbone
        oc=backbone.out_channels
        self.fpn=SimpleFPN(oc['C3'],oc['C4'],oc['C5'],fpn_out)
        self.head=SSDHead(fpn_out, num_anchors, num_classes+1)  # +1 background
    def forward(self,x):
        c3,c4,c5=self.backbone(x)
        feats=self.fpn(c3,c4,c5)
        return self.head(feats)   # cls[B,A,K], reg[B,A,4]

## 6. Sinh anchors (đúng thứ tự head: level → hàng → cột → anchor)

In [ ]:
def box_xyxy_to_xywh(b):
    x1,y1,x2,y2=b.unbind(-1); w=x2-x1; h=y2-y1
    return torch.stack([x1+0.5*w, y1+0.5*h, w, h],-1)
def box_xywh_to_xyxy(b):
    cx,cy,w,h=b.unbind(-1)
    return torch.stack([cx-0.5*w, cy-0.5*h, cx+0.5*w, cy+0.5*h],-1)
def iou(boxes1, boxes2):
    tl=torch.max(boxes1[:,None,:2],boxes2[:,:2]); br=torch.min(boxes1[:,None,2:],boxes2[:,2:])
    wh=(br-tl).clamp(min=0); inter=wh[...,0]*wh[...,1]
    a1=(boxes1[:,2]-boxes1[:,0])*(boxes1[:,3]-boxes1[:,1])
    a2=(boxes2[:,2]-boxes2[:,0])*(boxes2[:,3]-boxes2[:,1])
    return inter/(a1[:,None]+a2-inter+1e-7)

def build_anchors(cfg):
    S=cfg['img_size']; anchors=[]
    for base, stride in zip(cfg['anchor_bases'], cfg['strides']):
        fh=fw=S//stride
        wh=[]
        for sc in cfg['anchor_scales']:
            for r in cfg['anchor_ratios']:
                wh.append((base*sc*(r**0.5), base*sc/(r**0.5)))
        for y in range(fh):
            for x in range(fw):
                cx=(x+0.5)*stride; cy=(y+0.5)*stride
                for (w,h) in wh:
                    anchors.append([cx,cy,w,h])
    a=torch.tensor(anchors,dtype=torch.float32)
    return a, box_xywh_to_xyxy(a)   # xywh, xyxy

anchors_xywh, anchors_xyxy = build_anchors(CFG)
anchors_xywh=anchors_xywh.to(device); anchors_xyxy=anchors_xyxy.to(device)
print('Tổng anchors:', anchors_xywh.shape[0])  # 40*40*6 + 20*20*6 + 10*10*6 = 12600

## 7. Encode/decode + Matching + Multibox loss

In [ ]:
V1, V2 = CFG['variances']
def encode(gt_xyxy, anc_xywh):
    gt=box_xyxy_to_xywh(gt_xyxy)
    g_cxcy=(gt[:,:2]-anc_xywh[:,:2])/anc_xywh[:,2:]
    g_wh=torch.log(gt[:,2:]/anc_xywh[:,2:]+1e-9)
    return torch.cat([g_cxcy/V1, g_wh/V2],-1)
def decode(pred, anc_xywh):
    cxcy=pred[:,:2]*V1*anc_xywh[:,2:]+anc_xywh[:,:2]
    wh=torch.exp(pred[:,2:]*V2)*anc_xywh[:,2:]
    return box_xywh_to_xyxy(torch.cat([cxcy,wh],-1))

def match_targets(gt_xyxy, gt_labels):
    """Trả về cls_t[A] (0=bg, -1=ignore, 1..C=lớp), reg_t[A,4]."""
    A=anchors_xyxy.shape[0]
    cls_t=torch.zeros(A,dtype=torch.long,device=device)
    reg_t=torch.zeros(A,4,device=device)
    if gt_xyxy.numel()==0:
        return cls_t, reg_t
    gt_xyxy=gt_xyxy.to(device); gt_labels=gt_labels.to(device)
    ious=iou(anchors_xyxy, gt_xyxy)                 # [A,G]
    best_iou, best_gt=ious.max(1)                   # mỗi anchor -> gt tốt nhất
    # đảm bảo mỗi GT có ít nhất 1 anchor
    best_a_iou, best_a=ious.max(0)                  # mỗi gt -> anchor tốt nhất
    for g in range(gt_xyxy.shape[0]):
        best_gt[best_a[g]]=g; best_iou[best_a[g]]=1.0
    pos=best_iou>=CFG['pos_iou']
    neg=best_iou< CFG['neg_iou']
    ignore=(~pos)&(~neg)
    cls_t[pos]=gt_labels[best_gt[pos]]+1
    cls_t[ignore]=-1
    if pos.any():
        reg_t[pos]=encode(gt_xyxy[best_gt[pos]], anchors_xywh[pos])
    return cls_t, reg_t

def multibox_loss(cls_pred, reg_pred, gt_boxes, gt_labels):
    B, A, K = cls_pred.shape
    cls_t=torch.zeros(B,A,dtype=torch.long,device=device)
    reg_t=torch.zeros(B,A,4,device=device)
    for i in range(B):
        ct, rt=match_targets(gt_boxes[i], gt_labels[i])
        cls_t[i]=ct; reg_t[i]=rt
    pos=cls_t>0
    num_pos=pos.sum().clamp(min=1)
    # loc loss (chỉ positive)
    loc_loss=F.smooth_l1_loss(reg_pred[pos], reg_t[pos], reduction='sum') if pos.any() else reg_pred.sum()*0
    # conf loss + hard negative mining
    valid=cls_t>=0
    conf_all=F.cross_entropy(cls_pred.view(-1,K), cls_t.clamp(min=0).view(-1), reduction='none').view(B,A)
    conf_all=conf_all*valid.float()
    loss_c=conf_all.clone(); loss_c[pos]=0
    _, idx=loss_c.sort(1,descending=True)
    _, rank=idx.sort(1)
    num_pos_img=pos.sum(1,keepdim=True)
    num_valid_img=valid.sum(1,keepdim=True)
    num_neg=torch.clamp(CFG['neg_pos_ratio']*num_pos_img, max=(num_valid_img-num_pos_img).clamp(min=0))
    neg=rank<num_neg
    conf_loss=conf_all[pos].sum()+conf_all[neg].sum()
    return (loc_loss+conf_loss)/num_pos, loc_loss/num_pos, conf_loss/num_pos

## 8. Khởi tạo model (nạp backbone + đóng băng)

In [ ]:
def build_detector():
    if os.path.exists(CFG['backbone_ckpt']):
        ck=torch.load(CFG['backbone_ckpt'], map_location='cpu')
        bb=EELANBackbone(C_stem=ck['C_stem'], widths=tuple(ck['widths']), m=ck['m'], g=ck['g'])
        bb.load_state_dict(ck['backbone'])
        print('Đã nạp backbone từ', CFG['backbone_ckpt'])
    else:
        print('⚠ Không thấy backbone.pt -> khởi tạo backbone ngẫu nhiên (train end-to-end).')
        bb=EELANBackbone(C_stem=32, widths=(64,128,192,256), m=2.0, g=2)
        CFG['freeze_backbone']=False
    det=EELANDetector(bb, NUM_ANCHORS, NUM_CLASSES, fpn_out=256).to(device)
    if CFG['freeze_backbone']:
        for p in det.backbone.parameters(): p.requires_grad=False
        det.backbone.eval()
        print('Backbone đã ĐÓNG BĂNG.')
    return det

model=build_detector()
n_train=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Tham số train được: {n_train/1e6:.2f}M')

## 9. Self-test (kiểm tra shape trước khi train)

In [ ]:
with torch.no_grad():
    xb=torch.randn(2,3,CFG['img_size'],CFG['img_size']).to(device)
    cls_o, reg_o=model(xb)
    print('cls:', tuple(cls_o.shape), '| reg:', tuple(reg_o.shape), '| anchors:', anchors_xywh.shape[0])
    assert cls_o.shape[1]==anchors_xywh.shape[0], 'Số anchor không khớp số ô head!'
    assert cls_o.shape[2]==NUM_CLASSES+1
    # thử loss với GT giả
    gb=[torch.tensor([[50,50,150,150.]]), torch.zeros((0,4))]
    gl=[torch.tensor([0]), torch.zeros((0,),dtype=torch.long)]
    l,ll,lc=multibox_loss(cls_o,reg_o,gb,gl)
    print(f'loss={l.item():.3f} (loc={ll.item():.3f}, conf={lc.item():.3f})  ✓ shapes OK')

## 10. Huấn luyện + lưu weights

In [ ]:
def train_detector(cfg):
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params, lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    hist={'loss':[], 'loc':[], 'conf':[], 'val_loss':[]}
    best=1e9
    for ep in range(cfg['epochs']):
        model.train()
        if cfg['freeze_backbone']: model.backbone.eval()
        tl=tc=tloc=n=0; t0=time.time()
        for imgs, boxes, labels, _ in train_loader:
            imgs=imgs.to(device)
            cls_o, reg_o=model(imgs)
            loss, lloc, lconf=multibox_loss(cls_o, reg_o, boxes, labels)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 10.0); opt.step()
            bs=imgs.size(0); tl+=loss.item()*bs; tloc+=lloc.item()*bs; tc+=lconf.item()*bs; n+=bs
        sched.step()
        # val loss
        model.eval(); vl=vn=0
        with torch.no_grad():
            for imgs, boxes, labels, _ in val_loader:
                imgs=imgs.to(device)
                cls_o, reg_o=model(imgs)
                loss,_,_=multibox_loss(cls_o,reg_o,boxes,labels)
                vl+=loss.item()*imgs.size(0); vn+=imgs.size(0)
        vloss=vl/max(vn,1)
        hist['loss'].append(tl/n); hist['loc'].append(tloc/n); hist['conf'].append(tc/n); hist['val_loss'].append(vloss)
        if vloss<best:
            best=vloss
            torch.save({'model':model.state_dict(),'cfg':cfg,'classes':CLASSES,
                        'anchors_cfg':{k:cfg[k] for k in ['img_size','strides','anchor_bases','anchor_scales','anchor_ratios','variances']}},
                       cfg['best_ckpt'])
        print(f"Epoch {ep+1:02d}/{cfg['epochs']} | train {tl/n:.3f} (loc {tloc/n:.3f}/conf {tc/n:.3f}) | val {vloss:.3f} | {time.time()-t0:.1f}s")
    print('Best val loss:', round(best,4), '->', cfg['best_ckpt'])
    return hist

assert len(train_ds)>0, 'Chưa có dữ liệu train! Hãy build VOC + nạp panda ở Mục 2 trước.'
hist=train_detector(CFG)

# đồ thị loss
plt.figure(figsize=(7,4.5))
plt.plot(hist['loss'],label='train total'); plt.plot(hist['val_loss'],label='val total')
plt.plot(hist['loc'],'--',label='loc'); plt.plot(hist['conf'],'--',label='conf')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=.3); plt.title('Detection training loss')
plt.savefig(os.path.join(RESULTS,'det_loss_curve.png'),dpi=150,bbox_inches='tight'); plt.show()

## 11. Inference: decode + NMS → vẽ bbox + xuất CSV

In [ ]:
def nms(boxes, scores, iou_th=0.45):
    keep=[]; idxs=scores.argsort(descending=True)
    while idxs.numel()>0:
        i=idxs[0]; keep.append(i.item())
        if idxs.numel()==1: break
        ov=iou(boxes[i].unsqueeze(0), boxes[idxs[1:]]).squeeze(0)
        idxs=idxs[1:][ov<=iou_th]
    return keep

@torch.no_grad()
def detect(img_tensor, conf_th=None, iou_th=None):
    """img_tensor: [3,S,S] đã chuẩn hóa. Trả về boxes[N,4] xyxy(px), scores[N], labels[N] (0..C-1)."""
    conf_th=CFG['conf_th'] if conf_th is None else conf_th
    iou_th=CFG['nms_iou'] if iou_th is None else iou_th
    model.eval()
    cls_o, reg_o=model(img_tensor.unsqueeze(0).to(device))
    scores=F.softmax(cls_o[0],-1)       # [A,K]
    boxes_all=decode(reg_o[0], anchors_xywh).clamp(0, CFG['img_size'])
    out_b, out_s, out_l=[],[],[]
    for c in range(NUM_CLASSES):         # bỏ qua background (cột 0)
        sc=scores[:,c+1]
        m=sc>conf_th
        if m.sum()==0: continue
        b=boxes_all[m]; s=sc[m]
        k=nms(b, s, iou_th)
        out_b.append(b[k]); out_s.append(s[k]); out_l.append(torch.full((len(k),), c, device=device))
    if out_b:
        return torch.cat(out_b), torch.cat(out_s), torch.cat(out_l)
    return torch.zeros((0,4)), torch.zeros((0,)), torch.zeros((0,),dtype=torch.long)

COLORS=[(239,71,111),(17,138,178),(6,214,160)]
def draw_and_save(imgf, out_path, conf_th=None):
    im=Image.open(imgf).convert('RGB'); W,H=im.size; S=CFG['img_size']
    t=transforms.functional.to_tensor(im.resize((S,S),Image.BILINEAR)); t=(t-MEAN)/STD
    b,s,l=detect(t, conf_th)
    draw=ImageDraw.Draw(im); sx,sy=W/S,H/S; rows=[]
    for (x1,y1,x2,y2),sc,lab in zip(b.tolist(),s.tolist(),l.tolist()):
        X1,Y1,X2,Y2=x1*sx,y1*sy,x2*sx,y2*sy
        col=COLORS[int(lab)%3]
        draw.rectangle([X1,Y1,X2,Y2],outline=col,width=3)
        draw.text((X1+2,max(0,Y1-12)), f'{CLASSES[int(lab)]} {sc:.2f}', fill=col)
        rows.append([os.path.basename(imgf),CLASSES[int(lab)],f'{sc:.4f}',
                     round(X1,1),round(Y1,1),round(X2,1),round(Y2,1)])
    im.save(out_path)
    return rows

# Chạy inference trên toàn bộ tập val, xuất CSV + vẽ vài ảnh
os.makedirs(os.path.join(RESULTS,'det_vis'), exist_ok=True)
all_rows=[]
for i, imgf in enumerate(val_ds.img_files):
    out=os.path.join(RESULTS,'det_vis', os.path.basename(imgf)) if i<20 else None
    if out: all_rows+=draw_and_save(imgf,out)
    else:
        im=Image.open(imgf).convert('RGB'); W,H=im.size; S=CFG['img_size']
        t=transforms.functional.to_tensor(im.resize((S,S),Image.BILINEAR)); t=(t-MEAN)/STD
        b,s,l=detect(t); sx,sy=W/S,H/S
        for (x1,y1,x2,y2),sc,lab in zip(b.tolist(),s.tolist(),l.tolist()):
            all_rows.append([os.path.basename(imgf),CLASSES[int(lab)],f'{sc:.4f}',
                             round(x1*sx,1),round(y1*sy,1),round(x2*sx,1),round(y2*sy,1)])
with open(os.path.join(RESULTS,'det_predictions.csv'),'w',newline='') as f:
    w=csv.writer(f); w.writerow(['image','class','confidence','x1','y1','x2','y2']); w.writerows(all_rows)
print('Đã lưu det_predictions.csv (', len(all_rows), 'detections) + ảnh vẽ trong results/det_vis/')

## 12. Đánh giá — mAP@0.5 + Precision/Recall (VOC-style)

In [ ]:
@torch.no_grad()
def gather_detections_gt():
    """Thu dự đoán và GT trên tập val ở toạ độ ảnh 320 (không scale ngược để so khớp nhất quán)."""
    preds=defaultdict(list)   # class -> list (score, img_id, box)
    gts=defaultdict(list)     # class -> list (img_id, box, used_flag_index)
    gt_count=defaultdict(int)
    S=CFG['img_size']
    for img_id in range(len(val_ds)):
        t, boxes, labels, _=val_ds[img_id]
        b,s,l=detect(t)
        for bb,ss,ll in zip(b.tolist(),s.tolist(),l.tolist()):
            preds[int(ll)].append((ss,img_id,bb))
        for bb,ll in zip(boxes.tolist(), labels.tolist()):
            gts[int(ll)].append([img_id,bb,False]); gt_count[int(ll)]+=1
    return preds, gts, gt_count

def voc_ap(rec, prec):
    mrec=np.concatenate(([0.],rec,[1.])); mpre=np.concatenate(([0.],prec,[0.]))
    for i in range(len(mpre)-1,0,-1): mpre[i-1]=max(mpre[i-1],mpre[i])
    idx=np.where(mrec[1:]!=mrec[:-1])[0]
    return np.sum((mrec[idx+1]-mrec[idx])*mpre[idx+1])

def evaluate_map(iou_th=0.5):
    preds, gts, gt_count=gather_detections_gt()
    aps={}; prec_at={}; rec_at={}
    for c in range(NUM_CLASSES):
        P=sorted(preds[c], key=lambda z:-z[0])
        npos=gt_count[c]
        if npos==0:
            aps[CLASSES[c]]=float('nan'); continue
        tp=np.zeros(len(P)); fp=np.zeros(len(P))
        gt_by_img=defaultdict(list)
        for g in gts[c]: gt_by_img[g[0]].append(g)
        for i,(sc,img_id,box) in enumerate(P):
            gl=gt_by_img[img_id]; best=0.5; bj=-1
            bt=torch.tensor([box],dtype=torch.float32)
            for j,g in enumerate(gl):
                ov=iou(bt, torch.tensor([g[1]],dtype=torch.float32)).item()
                if ov>best and not g[2]: best=ov; bj=j
            if bj>=0 and best>=iou_th: gl[bj][2]=True; tp[i]=1
            else: fp[i]=1
        tp_c=np.cumsum(tp); fp_c=np.cumsum(fp)
        rec=tp_c/(npos+1e-9); prec=tp_c/(tp_c+fp_c+1e-9)
        aps[CLASSES[c]]=voc_ap(rec,prec)
        prec_at[CLASSES[c]]=prec[-1] if len(prec) else 0.0
        rec_at[CLASSES[c]]=rec[-1] if len(rec) else 0.0
    return aps, prec_at, rec_at

aps, prec_at, rec_at=evaluate_map(0.5)
valid_ap=[v for v in aps.values() if not (isinstance(v,float) and math.isnan(v))]
mAP=float(np.mean(valid_ap)) if valid_ap else 0.0
lines=['=== Object Detection — đánh giá tập val (IoU=0.5) ===']
for c in CLASSES:
    lines.append(f'{c:>6s}: AP={aps[c]:.4f} | precision={prec_at.get(c,0):.4f} | recall={rec_at.get(c,0):.4f}')
lines.append(f'mAP@0.5 = {mAP:.4f}')
report='\n'.join(lines); print(report)
with open(os.path.join(RESULTS,'det_metrics.txt'),'w',encoding='utf-8') as f: f.write(report+'\n')
print('\nĐã lưu results/det_metrics.txt')

---
### Hoàn tất Notebook 02
Đầu ra: `weights/best_det.pt`, `results/det_loss_curve.png`, `results/det_predictions.csv`,
`results/det_vis/*.jpg`, `results/det_metrics.txt`.

> Nếu mAP thấp: tăng `epochs`, bỏ đóng băng backbone (`freeze_backbone=False`), tăng số ảnh/lớp,
> hoặc chỉnh `anchor_bases` cho khớp kích thước vật thể.